# E-commerce shopping assistant example with small retrieval matching of products and skills

##Tried with e-commerce business approach with bunch of products and its minimal reteival for grounded suggestion in inventory

Included demo products list with links scrapped from web
Disclaimer: Sample links may restrict  
It's just a trail and error. Please free to review, suggest and correct 

In [2]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [3]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins gsk_qECw


In [4]:
# Initialize
groq_url = "https://api.groq.com/openai/v1"
openai = OpenAI(api_key=openai_api_key, base_url=groq_url)
MODEL = 'openai/gpt-oss-safeguard-20b'
history = []

In [5]:
products = [
  {
    "name": "Cat & Jack Kids Clothing",
    "url": "https://www.target.com/b/cat-jack/-/N-qqqgm",
    "section": "Kids",
    "description": "Target's in-house kids' apparel line covering newborn through tween sizes, backed by a one-year satisfaction guarantee on wear and tear.",
    "category": "clothing"
  },
  {
    "name": "Cat & Jack Boys' Clothing",
    "url": "https://www.target.com/c/boys-clothing-kids/cat-&-jack/-/N-5xty4Zqqqgm",
    "section": "Kids",
    "description": "Everyday boys' basics and standout styles built to grow with your child, with adjustable waistbands on many pants and jeans for extended wear.",
    "category": "clothing"
  },
  {
    "name": "Cat & Jack Girls' Clothing",
    "url": "https://www.target.com/c/girls-clothing-kids/cat-jack/-/N-5xtwaZqqqgm",
    "section": "Kids",
    "description": "Girls' denim shorts, skorts, leggings and everyday pieces designed for comfort and durability through active play.",
    "category": "clothing"
  },
  {
    "name": "Cat & Jack Toddler Clothing",
    "url": "https://www.target.com/c/toddler-clothing-kids/cat-jack/-/N-23cj2Zqqqgm",
    "section": "Kids",
    "description": "Toddler-sized shorts, tanks and tees made for easy movement and easy washing during the toughest play sessions.",
    "category": "clothing"
  },
  {
    "name": "Cat & Jack Baby Clothing",
    "url": "https://www.target.com/c/baby-clothing-kids/cat-jack/-/N-564plZqqqgm",
    "section": "Kids",
    "description": "Soft cotton sets and outfits sized 6–10 months and up, designed for newborn and infant comfort.",
    "category": "clothing"
  },
  {
    "name": "Kids' Christmas Sweaters (Old Navy)",
    "url": "https://oldnavy.gap.com/shop/kids-christmas-sweaters",
    "section": "Kids",
    "description": "Festive, durable kids' sweaters with playful graphics and 3D details, machine washable and made to survive holiday play.",
    "category": "sweater"
  },
  {
    "name": "Tommy Hilfiger Men's Modern-Fit Solid Navy Blazer",
    "url": "https://www.macys.com/shop/product/tommy-hilfiger-mens-modern-fit-solid-navy-blazer?ID=15914172",
    "section": "Adults",
    "description": "A modern-fit navy blazer with gold-tone buttons that dresses up for formal events or pairs down with jeans for a casual look.",
    "category": "blazer"
  },
  {
    "name": "Michael Kors Men's Classic-Fit Stretch Solid Blazer",
    "url": "https://www.macys.com/shop/product/michael-kors-mens-classic-fit-stretch-solid-blazers?ID=10671754",
    "section": "Adults",
    "description": "A streamlined, stretch-performance blazer with an athletic fit through the shoulders and chest, built for business-ready comfort.",
    "category": "blazer"
  },
  {
    "name": "Club Room Men's 100% Linen Blazer",
    "url": "https://www.macys.com/shop/product/club-room-mens-100-linen-blazer-created-for-macys?ID=12576115",
    "section": "Adults",
    "description": "A lightweight, breathable linen blazer with a classic tailored fit, ideal for warm-weather events and summer occasions.",
    "category": "blazer"
  },
  {
    "name": "Women's Formal Dresses & Evening Gowns (Nordstrom)",
    "url": "https://www.nordstrom.com/browse/women/clothing/dresses/formal-dresses",
    "section": "Adults",
    "description": "Nordstrom's curated edit of women's evening gowns and formal dresses from top brands, spanning silhouettes from A-line to halter.",
    "category": "dress"
  },
  {
    "name": "Old Navy Christmas Sweaters",
    "url": "https://oldnavy.gap.com/shop/christmas-sweaters",
    "section": "Adults",
    "description": "Cozy, high-quality adult holiday sweaters in crewneck and oversized fits, featuring festive graphics and knit patterns.",
    "category": "sweater"
  },
  {
    "name": "Mode of One Men's Slim-Fit Suit Blazer",
    "url": "https://www.macys.com/shop/product/mode-of-one-mens-slim-fit-suit-blazer-created-for-macys?ID=18077620",
    "section": "Adults",
    "description": "A versatile slim-fit suit blazer in a neutral tone, praised by reviewers for its modern style across both casual and formal occasions.",
    "category": "blazer"
  },
  {
    "name": "Nike Vomero 18",
    "url": "https://www.nike.com/w/mens-running-shoes-37v7jz5pxh3znik1zy7ok",
    "section": "Sports",
    "description": "Nike's best-selling men's road running shoe, built with maximum cushioning for everyday training miles.",
    "category": "shoe"
  },
  {
    "name": "Nike Pegasus Premium",
    "url": "https://www.nike.com/w/mens-shoes-nik1zy7ok",
    "section": "Sports",
    "description": "A premium road running shoe from Nike's long-standing Pegasus line, tuned for responsive daily-run cushioning.",
    "category": "shoe"
  },
  {
    "name": "Nike Alphafly 3",
    "url": "https://www.nike.com/w/mens-shoes-nik1zy7ok",
    "section": "Sports",
    "description": "Nike's top-tier road racing shoe, engineered for maximum energy return on race day.",
    "category": "shoe"
  },
  {
    "name": "Nike Air Force 1 '07",
    "url": "https://www.nike.com/w/mens-shoes-nik1zy7ok",
    "section": "Sports",
    "description": "The iconic Nike sneaker, a consistent best seller known for its durable leather build and versatile everyday styling.",
    "category": "shoe"
  },
  {
    "name": "Nike ACG Pegasus Trail",
    "url": "https://www.nike.com/w/new-mens-running-shoes-37v7jz3n82yznik1zy7ok",
    "section": "Sports",
    "description": "A men's trail running shoe built for off-road grip and stability, a current Nike best seller.",
    "category": "shoe"
  },
  {
    "name": "Nike Zoom Vomero 5",
    "url": "https://www.nike.com/w/mens-shoes-nik1zy7ok",
    "section": "Sports",
    "description": "A retro-inspired running shoe blending Nike's Vomero cushioning heritage with an everyday lifestyle look.",
    "category": "shoe"
  },
  {
    "name": "Cozy-Knit Holiday Sweater",
    "url": "https://oldnavy.gap.com/browse/product.do?pid=835916002",
    "section": "Holiday",
    "description": "A soft, vibrant crew-neck sweater with a front graphic, praised by customers for its cozy oversized fit and versatile styling.",
    "category": "sweater"
  },
  {
    "name": "Holiday Graphic Sweatshirt",
    "url": "https://oldnavy.gap.com/browse/product.do?pid=803227042",
    "section": "Holiday",
    "description": "A rib-knit pullover sweatshirt with a seasonal front graphic, popular as a gift and for matching family holiday outfits.",
    "category": "sweatshirt"
  },
  {
    "name": "Matching Holiday Fair Isle Cardigan Sweater",
    "url": "https://oldnavy.gap.com/browse/product.do?pid=477619002",
    "section": "Holiday",
    "description": "A five-button Fair Isle cardigan designed for matching family sets, with a rib-knit crew neck and cuffs for a classic holiday look.",
    "category": "cardigan"
  },
  {
    "name": "Men's Christmas Sweaters (Old Navy)",
    "url": "https://oldnavy.gap.com/shop/mens-christmas-sweaters",
    "section": "Holiday",
    "description": "Soft, breathable men's holiday sweaters in classic crewneck and pullover styles, easy to pair with jeans or chinos.",
    "category": "sweater"
  },
  {
    "name": "Women's Christmas Sweaters (Old Navy)",
    "url": "https://oldnavy.gap.com/shop/womens-christmas-sweaters",
    "section": "Holiday",
    "description": "Festive women's sweaters in vibrant colors and patterns, made from soft breathable fabric for indoor and outdoor holiday wear.",
    "category": "sweater"
  },
  {
    "name": "Holiday-Graphic Sweatshirt",
    "url": "https://oldnavy.gap.com/browse/product.do?pid=486170022",
    "section": "Holiday",
    "description": "A relaxed-fit sweatshirt featuring a seasonal graphic print, designed for casual holiday gatherings.",
    "category": "sweatshirt"
  },
  {
    "name": "Men's Suit Blazers & Sport Coats (Macy's)",
    "url": "https://www.macys.com/shop/mens/clothing/blazers-sport-coats/Blazer_style/Suit?id=16499",
    "section": "Formals",
    "description": "Macy's full collection of men's suit blazers and sport coats from brands like Tommy Hilfiger, Michael Kors, and Club Room, for formal events and business wear.",
    "category": "blazer"
  },
  {
    "name": "Men's Modern Blazers & Sport Coats (Macy's)",
    "url": "https://www.macys.com/shop/mens-clothing/all-mens-clothing/mens-blazers-sports-coats/Suit_fit/Modern?id=16499",
    "section": "Formals",
    "description": "Modern-fit tailored blazers and sport coats designed to polish off a formal ensemble for any occasion.",
    "category": "blazer"
  },
  {
    "name": "Women's Formal Dresses (Nordstrom)",
    "url": "https://www.nordstrom.com/browse/women/clothing/dresses?filterByOccasion=formal",
    "section": "Formals",
    "description": "Nordstrom's formal-occasion dress edit spanning cocktail dresses to full gowns, curated for weddings and events.",
    "category": "dress"
  },
  {
    "name": "Women's Black Formal Dresses & Evening Gowns (Nordstrom)",
    "url": "https://www.nordstrom.com/browse/women/clothing/dresses/formal-dresses?filterByColor=black",
    "section": "Formals",
    "description": "A classic black evening gown edit at Nordstrom, spanning multiple silhouettes and designer brands for black-tie events.",
    "category": "dress"
  },
  {
    "name": "Big & Tall Blazers & Sport Coats (Macy's)",
    "url": "https://www.macys.com/shop/mens/clothing/big-and-tall/blazers-sport-coats?id=339894",
    "section": "Formals",
    "description": "Formal blazers and sport coats sized specifically for big and tall fits, for a tailored look at formal occasions.",
    "category": "blazer"
  },
  {
    "name": "Women's Alex Evenings Dresses (Nordstrom)",
    "url": "https://www.nordstrom.com/browse/women/clothing/dresses?filterByBrand=alex-evenings",
    "section": "Formals",
    "description": "A dedicated collection from formalwear specialist Alex Evenings, featuring gowns and cocktail dresses for entrances, dances, and galas.",
    "category": "dress"
  }
]

In [7]:
system_message = f"""
  You are a kind,warm and helpful assistant in a clothes store. where store has three sections:Kids,Adults,Sports,HolidayFormals

  You should suggest only from products list below and its detail aspects in the store for the customers comes for occasional shopping. 
  For example, if the customer says 'I'm looking for trousers to travel', you could reply something like, 
  'Wonderful - we have lots of trousers in the mens section for travel, you came at right time, summer sale starts'
"""

In [ ]:
def skill():
    return """Format the description by making every matching product name bold, underlined, and hyperlinked to its product URL, and making every matching search query term bold and underlined (no link).
     Matching should be case-insensitive and whole-word, with longer product names matched before shorter ones to avoid partial overlaps. 
    Product listing should only in bullets. No tables or flowchart 
    If a product name and the search query overlap in the same text, keep the product name's bold+underline+link formatting and don't re-wrap it for the query.
     Leave all other text in the description unchanged."""

In [9]:
def products_match(message):
    msg = f"""user_query: {message}"""
    relevant_products = [q for q in products if q['category'] or q['name'] in message]
    if len(relevant_products) > 0 :
     msg += f"These are the relevant products in store below:{relevant_products}"
    return msg
    # else: 
    #   return system_message + "understand the requirments clearly"


In [12]:
def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message  = system_message + products_match(message)

    messages = [{"role": "system", "content": relevant_system_message}] + history +[{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [13]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7889
* To create a public link, set `share=True` in `launch()`.
